In [22]:
import pandas as pd

In [23]:
filepath = "nrc_data_csv/ukrainian_nrc.csv"
df = pd.read_csv(filepath)
df.head(12)

,word,anger,disgust,fear,joy,sadness,surprise
0,назад,0,0,0,0,0,0
1,рахівниця,0,0,0,0,0,0
2,відмовитися,0,0,1,0,1,0
3,покинутий,1,0,1,0,1,0
4,занедбаність,1,0,1,0,1,1
5,зменшуватися,0,0,0,0,0,0
6,зменшення,0,0,0,0,0,0
7,abba,0,0,0,0,0,0
8,настоятель,0,0,0,0,0,0
9,скорочувати,0,0,0,0,0,0


In [24]:
from datasets import load_dataset
import pandas as pd

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
random_state = 42

# Load German evaluation data

comb_df = pd.concat(
    map(pd.read_csv, ["brighter_ukranian.csv"]),
    ignore_index=True
)

comb_df["text"] = comb_df["text"].astype(str)
comb_df["emotion"] = comb_df["emotion"].astype(str)

print("Original German rows:", len(comb_df))
print("\nOriginal German class distribution:")
print(comb_df["emotion"].value_counts())


# Balance German data
target_n = comb_df["emotion"].value_counts().min()

balanced_parts = []
for emotion, group in comb_df.groupby("emotion"):
    balanced_parts.append(
        group.sample(n=target_n, random_state=random_state)
    )

comb_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("\nBalanced German rows:", len(comb_df))
print("\nBalanced German class distribution:")
print(comb_df["emotion"].value_counts())

Original German rows: 3583

Original German class distribution:
emotion
joy         1185
sadness      936
surprise     527
fear         479
anger        250
disgust      206
Name: count, dtype: int64

Balanced German rows: 1236

Balanced German class distribution:
emotion
disgust     206
joy         206
surprise    206
anger       206
fear        206
sadness     206
Name: count, dtype: int64


In [25]:
import spacy
nlp = spacy.load("uk_core_news_sm")

print(df)

               word  anger  disgust  fear  joy  sadness  surprise
0             назад      0        0     0    0        0         0
1         рахівниця      0        0     0    0        0         0
2       відмовитися      0        0     1    0        1         0
3         покинутий      1        0     1    0        1         0
4      занедбаність      1        0     1    0        1         1
...             ...    ...      ...   ...  ...      ...       ...
14149          зона      0        0     0    0        0         0
14150       зоопарк      0        0     0    0        0         0
14151   зоологічний      0        0     0    0        0         0
14152      зоології      0        0     0    0        0         0
14153     збільшити      0        0     0    0        0         0

[14154 rows x 7 columns]


In [26]:
import random

random.seed(42)

df["word"] = df["word"].astype(str).str.lower().str.strip()

df = (
    df.groupby("word", as_index=False)[target_emotions]
    .max()
)

lexicon = df.set_index("word")[target_emotions].to_dict(orient="index")

emotion_frequency = {}

for word, scores in lexicon.items():
    active_count = sum(scores[emotion] for emotion in target_emotions)

    if active_count > 0:
        emotion_frequency[word] = 1 / active_count
    else:
        emotion_frequency[word] = 0

predictions = []

for text in comb_df["text"]:

    count = {emotion: 0 for emotion in target_emotions}

    doc = nlp(text)

    for token in doc:

        candidates = [
            token.lemma_.lower().strip() if token.lemma_ else "",
            token.text.lower().strip()
        ]

        for candidate in candidates:

            if candidate in lexicon:

                word_weight = emotion_frequency[candidate]

                for emotion in target_emotions:
                    count[emotion] += (
                        lexicon[candidate][emotion] * word_weight
                    )

                break

    maxim = max(count.values())

    print(text)
    print(count)
    print()

    if maxim == 0:
        pred = "unknown"
    else:
        best_emotions = [emo for emo, score in count.items() if score == maxim]
        pred = random.choice(best_emotions)

    predictions.append(pred)

comb_df["prediction"] = predictions


кроси страшні як я завтра зранку
{'anger': 0.2, 'disgust': 0.2, 'fear': 0.2, 'joy': 0.0, 'sadness': 0.2, 'surprise': 0.2}

з днем народження, кицю   щастя тобі :3 кохай і будь коханою
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.5, 'joy': 2.5, 'sadness': 0.0, 'surprise': 0.0}

о боги,  так ти з бц,  ми випадково не пересікались у реалі?
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.0, 'sadness': 0.0, 'surprise': 1.0}

я така зла, ви навіть не уявляєте.
{'anger': 0.3333333333333333, 'disgust': 0.3333333333333333, 'fear': 0.3333333333333333, 'joy': 0.0, 'sadness': 0.0, 'surprise': 0.0}

так рознервувалась що не помітила як злобно проорала абзац з книжки вслух
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

ти приїхала у кременець?)) я почала хвилюватися;)
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

сьогодні день поцілунків, тому я йду цілувати свого котіка
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.5, 'sadness':

In [27]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def evaluate_lexicon_predictions(df, gold_col="emotion", pred_col="prediction"):
    """
    Evaluates lexicon predictions.
    Unknown predictions are counted as wrong in the main evaluation.
    Also reports coverage and covered-only performance.
    """

    eval_df = df.copy()
    eval_df[gold_col] = eval_df[gold_col].astype(str)
    eval_df[pred_col] = eval_df[pred_col].astype(str)

    # -------------------------
    # Main evaluation: unknown = wrong
    # -------------------------
    y_true = eval_df[gold_col]
    y_pred = eval_df[pred_col]

    print("=== Main evaluation: unknown counted as wrong ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="weighted"), 4))
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=target_emotions,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    cm_df = pd.DataFrame(cm, index=target_emotions, columns=target_emotions)

    print("Confusion matrix:")
    print(cm_df)
    print()

    # -------------------------
    # Coverage
    # -------------------------
    covered_mask = eval_df[pred_col] != "unknown"
    coverage = covered_mask.mean()

    print("=== Coverage ===")
    print("Coverage:", round(coverage, 4))
    print("Covered examples:", covered_mask.sum())
    print("Total examples:", len(eval_df))
    print()

    # -------------------------
    # Covered-only evaluation
    # -------------------------
    covered_df = eval_df[covered_mask].copy()

    if len(covered_df) > 0:
        print("=== Covered-only evaluation ===")
        print("Accuracy:", round(accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Balanced accuracy:", round(balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Macro F1:", round(f1_score(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            average="macro",
            zero_division=0
        ), 4))
        print()

        print(classification_report(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            digits=3,
            zero_division=0
        ))
    else:
        print("No covered examples.")

    results = {
        "accuracy_unknown_wrong": accuracy_score(y_true, y_pred),
        "balanced_accuracy_unknown_wrong": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="macro", zero_division=0),
        "weighted_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="weighted", zero_division=0),
        "coverage": coverage,
    }

    if len(covered_df) > 0:
        results.update({
            "accuracy_covered_only": accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "balanced_accuracy_covered_only": balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "macro_f1_covered_only": f1_score(
                covered_df[gold_col],
                covered_df[pred_col],
                labels=target_emotions,
                average="macro",
                zero_division=0
            ),
        })

    return results

In [28]:
results = evaluate_lexicon_predictions(
    comb_df,
    gold_col="emotion",
    pred_col="prediction"
)

=== Main evaluation: unknown counted as wrong ===
Accuracy: 0.1626
Balanced accuracy: 0.1626
Macro F1: 0.2169
Weighted F1: 0.2169

              precision    recall  f1-score   support

       anger      0.278     0.097     0.144       206
     disgust      0.459     0.165     0.243       206
        fear      0.182     0.107     0.135       206
         joy      0.344     0.272     0.304       206
     sadness      0.456     0.199     0.277       206
    surprise      0.373     0.136     0.199       206

   micro avg      0.338     0.163     0.220      1236
   macro avg      0.349     0.163     0.217      1236
weighted avg      0.349     0.163     0.217      1236

Confusion matrix:
          anger  disgust  fear  joy  sadness  surprise
anger        20        7    30   18       13         7
disgust      21       34    20   12       11         8
fear          8        7    22   25        9        12
joy           3        0    14   56        6        17
sadness      12       11    24   

/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
